In [1]:
import os
import cv2
import time
import pandas as pd
from IPython.display import display, Image, clear_output

# Create directories
folders = ['dataset/open', 'dataset/closed']
for f in folders:
    os.makedirs(f, exist_ok=True)

def get_counts():
    return {
        'open': len(os.listdir('dataset/open')),
        'closed': len(os.listdir('dataset/closed'))
    }

In [ ]:
cap = cv2.VideoCapture(1)

print("Starting webcam... Please focus on the OpenCV popup window to use keyboard controls.")

last_saved_class = "None"
frame_count = 0

while True:
    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame.")
        break
        
    frame_copy = frame.copy()
    cv2.putText(frame_copy, "Focus here for keys (q/o/c)", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    
    cv2.imshow("Webcam Feed - Focus here for keyboard", frame_copy)
    key = cv2.waitKey(1) & 0xFF
    
    saved = False
    if key == ord('q'):
        break
    elif key == ord('o'):
        count = len(os.listdir('dataset/open'))
        filename = f"dataset/open/open_{count+1:03d}.jpg"
        cv2.imwrite(filename, frame)
        last_saved_class = f"Open ({filename})"
        saved = True
    elif key == ord('c'):
        count = len(os.listdir('dataset/closed'))
        filename = f"dataset/closed/closed_{count+1:03d}.jpg"
        cv2.imwrite(filename, frame)
        last_saved_class = f"Closed ({filename})"
        saved = True
    frame_count += 1
    if saved or frame_count % 30 == 0:
        clear_output(wait=True)
        counts = get_counts()
        print("====== LIVE DATA COLLECTION ======")
        print(f"Open: {counts['open']} | Closed: {counts['closed']}")
        print(f"Total photos: {sum(counts.values())} / 600 target")
        print(f"Last saved: {last_saved_class}")
        print("==================================")
        
        _, ret_img_live = cv2.imencode('.jpg', frame)
        display(Image(data=ret_img_live.tobytes(), width=400, format='jpeg'))

cap.release()
cv2.destroyAllWindows()

In [ ]:
#summary
counts = get_counts()
summary = []
for cat, cnt in counts.items():
    samples = os.listdir(f"dataset/{cat}")[:3] if os.path.exists(f"dataset/{cat}") else []
    summary.append({'class_name': cat, 'total_photos': cnt, 'sample_files': str(samples)})

df = pd.DataFrame(summary)
df.to_csv("collection_summary.csv", index=False)
print("Data collection completed and saved to collection_summary.csv")
display(df)

Data collection completed and saved to collection_summary.csv


,class_name,total_photos,sample_files
0,open,900,"['open_776.jpg', 'open_010.jpg', 'open_004.jpg']"
1,closed,866,"['closed_338.jpg', 'closed_304.jpg', 'closed_4..."
